<a href="https://colab.research.google.com/github/valdinilemofouet-ui/LLM-Alignment-for-LRL/blob/main/LLM_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Set up

In [1]:
!pip install -q transformers accelerate bitsandbytes datasets peft
!pip install -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00


In [3]:
from huggingface_hub import notebook_login
notebook_login()

## Evaluation Script

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from datasets import load_dataset
import json

# 1. Configuration
# Change 'model_id' to "meta-llama/Meta-Llama-3.1-8B-Instruct" for Llama
model_id = "AfriSenti/Afriqueqwen-8B" # Update with the exact HF path for Afriqueqwen
language_code = "ha" # Hausa

# 2. Load UbuntuGuard Dataset (Hausa)
# Assuming the data is hosted on HF or cloned from the GitHub repo
dataset = load_dataset("hemhemoh/UbuntuGuard", split="test")
hausa_data = dataset.filter(lambda x: x['language'] == 'ha')

# 3. Load Model in 4-bit for Colab efficiency
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config={"load_in_4bit": True},
    torch_dtype=torch.bfloat16
)

# 4. Define Inference Function
def evaluate_safety(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]
    # Formatting for Chat Models
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.1 # Low temperature for consistency
    )

    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return response

# 5. Run Evaluation
results = []
for item in hausa_data.select(range(50)): # Start with a sample of 50
    prompt = item['prompt']
    category = item['category']
    print(f"Testing Category: {category}")

    response = evaluate_safety(prompt)

    results.append({
        "prompt": prompt,
        "category": category,
        "model_response": response
    })

# 6. Save Results
with open(f"{model_id.split('/')[-1]}_hausa_results.json", "w", encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)